# Notebook 1: Introduction to the Gulf Crest Case

**Module:** AI-Augmented Catastrophe Claims Management
**Company:** Gulf Crest Insurance Company (fictional)
**Event:** Hurricane Marisol (fictional, Category 4, made landfall July 18, 2026)

### The situation

Gulf Crest Insurance Company sells home and business property insurance along a coastline. About 30 days
ago, Hurricane Marisol made landfall as a strong Category 4 storm and caused widespread damage across
Gulf Crest's coastal policyholders. Thousands of insurance claims have come in, and you, playing the role
of Gulf Crest's actuarial analysis team, have been asked to help make sense of it all.

Over the next several notebooks, you will:
1. Explore the data (this notebook + Notebook 2)
2. Predict how much claims will ultimately cost (Notebook 3)
3. Flag claims that might be fraudulent (Notebook 4)
4. Estimate how much money the company needs to set aside to pay for these claims (Notebook 5)
5. Decide how to prioritize claims given limited staff (Notebook 6)
6. Test "what if" scenarios (Notebook 7)
7. Write a short recommendation for company leadership (Notebook 8)

This first notebook establishes what data is available and what it looks like.

In [1]:
# We start by "importing" the tools we'll use throughout this module.
# Think of each import as picking up a tool from a toolbox before starting work.

import pandas as pd   # pandas lets us work with tables of data (like Excel spreadsheets, but in code)
import numpy as np    # numpy helps with numbers and calculations

# This makes tables print more legibly in the notebook - a cosmetic setting only
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Setup complete. Ready to load data.")

Setup complete. Ready to load data.


## Loading the Datasets

There are 8 datasets in this module, each describing a different part of Gulf Crest's business. A dataset
here is a table (rows and columns) saved as a `.csv` file, the same kind of file Excel can open.

We use `pandas` (imported above as `pd`) to load each one into what's called a **DataFrame**, think of a
DataFrame as a spreadsheet that lives inside the notebook, which we can then filter, sort, and analyze with
code instead of mouse clicks.

In [2]:
# The datasets live one folder up, inside "04_Datasets".
# We load each CSV file into its own DataFrame (its own "spreadsheet" inside the notebook).

DATA_DIR = "../04_Datasets/"

policies          = pd.read_csv(DATA_DIR + "01_policies.csv")
claims            = pd.read_csv(DATA_DIR + "02_claims.csv")
weather           = pd.read_csv(DATA_DIR + "03_weather.csv")
weather_zones     = pd.read_csv(DATA_DIR + "03_weather_zone_impact.csv")
economic          = pd.read_csv(DATA_DIR + "04_economic_indicators.csv")
repair_costs      = pd.read_csv(DATA_DIR + "05_repair_costs.csv")
fraud_labels      = pd.read_csv(DATA_DIR + "06_fraud_labels.csv")
reserve_history   = pd.read_csv(DATA_DIR + "07_reserve_history.csv")
financials        = pd.read_csv(DATA_DIR + "08_company_financials.csv")

print("All 8 datasets loaded successfully.")

All 8 datasets loaded successfully.


## 1. Policies: Who Does Gulf Crest Insure?

A **policy** is a contract: a customer pays Gulf Crest a yearly amount (the "premium"), and in exchange
Gulf Crest promises to pay out if something covered happens to their property (like storm damage).

The next cell shows the first few rows. `.head()` displays only the top of the table rather than the entire dataset.

In [3]:
policies.head()

,policy_id,policyholder_name,property_address,zone_id,coastal_zone_flag,construction_type,year_built,coverage_type,total_insured_value,policy_start_date,annual_premium
0,GC-000001,Allison Hill,"18196 Anthony Forge, Marsh Point, Belmara County",Z2,Y,Superior (Reinforced Concrete),1999,Homeowners,534800.0,2022-04-12,2181.50
1,GC-000002,Lance Hoffman,"79402 Peterson Drives Apt. 511, Port Alden, Be...",Z1,Y,Masonry,1972,Homeowners,237300.0,2024-08-25,1270.23
2,GC-000003,Ian Cooper,"78161 Calderon River Suite 931, Millbrook, Bel...",Z6,N,Masonry Veneer,1991,Homeowners,226300.0,2022-04-05,805.99
3,GC-000004,Darren Roberts,"16475 Mitchell Fords, Ridgeview, Belmara County",Z4,N,Frame,2000,Homeowners,235700.0,2025-02-08,888.19
4,GC-000005,Mia Sutton,"283 Steven Groves, Millbrook, Belmara County",Z6,N,Frame,2003,Homeowners,204300.0,2024-06-21,760.85


In [4]:
# .shape tells us (number of rows, number of columns) - i.e. how many policies, and how many pieces
# of information we have about each one.
print("Number of policies:", policies.shape[0])
print("Number of columns (fields) per policy:", policies.shape[1])

# .info() gives a quick summary: column names, how many non-empty values each has, and their data type
policies.info()

Number of policies: 5000
Number of columns (fields) per policy: 11
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   policy_id            5000 non-null   str    
 1   policyholder_name    5000 non-null   str    
 2   property_address     5000 non-null   str    
 3   zone_id              5000 non-null   str    
 4   coastal_zone_flag    5000 non-null   str    
 5   construction_type    5000 non-null   str    
 6   year_built           5000 non-null   int64  
 7   coverage_type        5000 non-null   str    
 8   total_insured_value  5000 non-null   float64
 9   policy_start_date    5000 non-null   str    
 10  annual_premium       5000 non-null   float64
dtypes: float64(2), int64(1), str(8)
memory usage: 429.8 KB


Notice the `total_insured_value` column: this is how much the property is insured for. And
`coastal_zone_flag` tells us whether the property sits in one of the coastal zones that took the brunt of
Hurricane Marisol.

The next cell shows the split between coastal and inland policies, and between homeowners and commercial coverage.

In [5]:
# .value_counts() counts how many times each unique value appears in a column.
# This is one of the most useful pandas functions you'll use throughout this module.

print("Coastal vs inland policies:")
print(policies["coastal_zone_flag"].value_counts())

print("\nCoverage type split:")
print(policies["coverage_type"].value_counts())

print("\nPolicies per zone:")
print(policies["zone_id"].value_counts())

Coastal vs inland policies:
coastal_zone_flag
Y    2961
N    2039
Name: count, dtype: int64

Coverage type split:
coverage_type
Homeowners    4111
Commercial     889
Name: count, dtype: int64

Policies per zone:
zone_id
Z1    1102
Z2     957
Z3     902
Z4     729
Z6     684
Z5     626
Name: count, dtype: int64


## 2. Claims: What Happened After the Storm?

A **claim** is filed when a policyholder asks Gulf Crest to pay for damage. Not every policy has a claim; only policyholders who suffered a loss file one.

Each claim links back to a policy through the `policy_id` column. This is the same kind of relationship
you'd see between two related tables in any database (a "foreign key," if you've done any database work
before; if not, think of it as claims "pointing back" to the policy they belong to).

In [6]:
claims.head()

,claim_id,policy_id,date_of_loss,date_reported,cause_of_loss,claim_status,is_catastrophe_flag,storm_id,reported_severity,settlement_date
0,CLM-000001,GC-000931,2021-03-15,2021-03-21,Fire,Closed,N,NaN,8689.85,2021-05-27
1,CLM-000002,GC-003769,2021-04-22,2021-04-28,Vandalism,Closed,N,NaN,858.40,2021-05-04
2,CLM-000003,GC-001968,2021-05-19,2021-05-20,Liability,Closed,N,NaN,10464.68,2021-06-04
3,CLM-000004,GC-000149,2021-05-30,2021-05-30,Fire,Closed,N,NaN,25062.11,2021-07-03
4,CLM-000005,GC-001462,2021-06-01,2021-06-06,Liability,Closed,N,NaN,13536.78,2021-06-27


In [7]:
print("Total number of claims:", claims.shape[0])
print("\nCatastrophe (Hurricane Marisol) vs routine claims:")
print(claims["is_catastrophe_flag"].value_counts())

print("\nClaim status (open = still being processed, closed = fully paid and finished):")
print(claims["claim_status"].value_counts())

Total number of claims: 2883

Catastrophe (Hurricane Marisol) vs routine claims:
is_catastrophe_flag
Y    2021
N     862
Name: count, dtype: int64

Claim status (open = still being processed, closed = fully paid and finished):
claim_status
Open      2035
Closed     848
Name: count, dtype: int64


Notice how many more claims are `is_catastrophe_flag = Y` (from Marisol) compared to routine claims. That surge is exactly the challenge Gulf Crest's actuarial team is facing right now. Also notice that most
catastrophe claims are still `Open`. Only 30 days have passed since landfall, so most haven't been fully
settled yet. This is an important idea we'll come back to in Notebook 5 (Reserve Estimation): when an event
just happened, we don't yet know the final cost of most claims, so we have to *estimate* it.

In [8]:
# reported_severity is the dollar amount of the claim as currently estimated.
# .describe() gives us a quick statistical summary: count, average, spread, min/max, etc.
claims["reported_severity"].describe()

count    2.883000e+03
mean     6.193722e+04
std      1.403692e+05
min      9.087000e+01
25%      9.020735e+03
50%      2.289103e+04
75%      5.616783e+04
max      3.258045e+06
Name: reported_severity, dtype: float64

Look at the difference between the `mean` (average) and the `50%` (median, i.e. the middle value if you
lined up every claim from smallest to largest). The mean is usually noticeably higher than the median here.
That's a signal that a small number of very large claims are pulling the average upward, a pattern called
a **"long tail"** or **"right-skewed" distribution**. Property damage claims are like this in real life too:
most claims are modest, but a handful of severe ones (e.g. total losses) are enormous. We'll visualize this
directly in Notebook 2.

## 3. Weather: What Was Hurricane Marisol Like?

This dataset tracks the storm itself: its position, wind speed, and strength over time, recorded roughly
every 6 hours as it moved. The `03_weather_zone_impact.csv` file summarizes how hard each of Gulf Crest's
6 geographic zones was hit.

In [9]:
weather.head()

,storm_id,storm_name,observation_datetime,latitude,longitude,max_sustained_wind_mph,central_pressure_mb,storm_category,is_landfall
0,MARISOL2026,Marisol,2026-07-13,21.94,-87.99,38.4,998.6,Tropical Depression,N
1,MARISOL2026,Marisol,2026-07-13,22.28,-88.02,41.1,996.3,Tropical Storm,N
2,MARISOL2026,Marisol,2026-07-13,22.51,-88.11,49.1,989.5,Tropical Storm,N
3,MARISOL2026,Marisol,2026-07-13,22.62,-88.28,56.8,NaN,Tropical Storm,N
4,MARISOL2026,Marisol,2026-07-14,22.89,-88.35,61.4,NaN,Tropical Storm,N


In [10]:
weather_zones

,storm_id,zone_id,zone_name,peak_wind_experienced_mph,storm_surge_ft
0,MARISOL2026,Z1,Port Alden,132,11.2
1,MARISOL2026,Z2,Marsh Point,138,13.5
2,MARISOL2026,Z3,Cape Verrin,121,8.7
3,MARISOL2026,Z4,Ridgeview,79,NaN
4,MARISOL2026,Z5,Hollow Bay,68,NaN
5,MARISOL2026,Z6,Millbrook,61,NaN


Notice that Z2 (Marsh Point) experienced the highest wind speed (138 mph) and the highest storm surge
(13.5 feet), and if you look back at the claims data, you'd expect Marsh Point policyholders to have filed
the most claims relative to their zone's policy count. We'll check that expectation directly in Notebook 2.

Also notice: the inland zones (Z4, Z5, Z6) have an empty (`NaN`) value for `storm_surge_ft`. That's not a
mistake: storm surge is a coastal phenomenon, so there's genuinely no surge measurement to report inland.
You'll see missing values like this throughout the datasets in this module. Real-world data is never
perfectly complete, and part of your job as an analyst is deciding what a missing value *means* before
deciding what to do about it.

## 4. The Remaining Datasets: A Quick Look

We won't dig deeply into these last four yet since they matter more in later notebooks, but let's confirm
they loaded correctly and get a first look at each.

In [11]:
print("Economic Indicators (monthly, region-wide):")
display(economic.head(3))

print("\nRepair Costs (line items per claim):")
display(repair_costs.head(3))

print("\nFraud Labels (is a claim believed to be fraudulent?):")
display(fraud_labels.head(3))

print("\nReserve History (how a claim's estimated cost changes over time):")
display(reserve_history.head(3))

print("\nCompany Financials (Gulf Crest's quarterly financial results):")
display(financials.tail(3))

Economic Indicators (monthly, region-wide):


,date,region,cpi_index,construction_cost_index,wage_index
0,2021-01-01,Belmara County,100.18,100.46,100.27
1,2021-02-01,Belmara County,100.43,100.72,100.48
2,2021-03-01,Belmara County,100.45,101.07,100.72



Repair Costs (line items per claim):


,claim_id,repair_category,estimated_cost,actual_cost,cost_date
0,CLM-000001,Plumbing,11178.77,13167.26,2021-05-27
1,CLM-000002,Electrical,659.76,740.92,2021-05-04
2,CLM-000004,Contents,25853.55,26154.24,2021-07-03



Fraud Labels (is a claim believed to be fraudulent?):


,claim_id,is_fraud,fraud_indicator_notes
0,CLM-000001,0.0,"Reviewed, no fraud indicators found"
1,CLM-000002,1.0,Confirmed fraud indicators found in review
2,CLM-000003,0.0,"Reviewed, no fraud indicators found"



Reserve History (how a claim's estimated cost changes over time):


,claim_id,evaluation_date,development_month,reserve_estimate,cumulative_paid
0,CLM-000001,2021-04-20,1,3634.62,1076.47
1,CLM-000001,2021-05-20,2,1647.88,1868.89
2,CLM-000001,2021-06-19,3,2681.26,3003.95



Company Financials (Gulf Crest's quarterly financial results):


,period,gross_written_premium,earned_premium,incurred_losses,loss_adjustment_expenses,policyholder_surplus
20,2026Q1,13859402.22,13266794.01,8380178.20,841341.99,88760970.59
21,2026Q2,14028932.75,13277542.48,8594078.50,1149291.69,90881473.96
22,2026Q3,14211274.03,13832172.23,44540208.26,5232963.83,69316874.04


Notice the last row(s) of `financials`: look at the `incurred_losses` column around the most recent
quarter compared to earlier ones. This is Hurricane Marisol showing up directly in the company's financial
statements. Keep this in mind: everything you calculate in Notebooks 3 through 8 ultimately feeds into numbers like these.

## Summary

You've now loaded and previewed all 8 datasets that make up this case. Before moving to Notebook 2 (EDA),
make sure you can answer these questions (see Workbook Activity 1):

1. Roughly what fraction of Gulf Crest's policies are in coastal zones?
2. How many claims resulted from Hurricane Marisol, versus routine (non-storm) causes?
3. Why are most catastrophe claims still "Open" rather than "Closed"?
4. Why might a dataset have a missing value that *isn't* a data-quality problem?

**Next:** Open `02_EDA.ipynb` to dig deeper into these datasets: distributions, missing data, and the
relationships between them.